# 365 Probabilidades - Dia #165
## Qual a probabilidade de você ficar mais de 11 minutos numa tarefa?

**Tipo:** Comportamental · Tempo  
**Data de publicação:** 2026-11-25  
**Ferramenta:** Python  
**Decisão analisada:** Quanto tempo reservar para uma tarefa que exige continuidade?  
**Hashtag:** #365Probabilidades #Dia165 #ProbabilidadesDoTempo

---

### 📖 A História

Você abre o e-mail para responder uma mensagem. Antes de terminar, chega um aviso no celular. Alguém chama para uma pergunta rápida. Quando você volta, já não lembra o que ia escrever.

Quase todo mundo conhece esse dia.

Fui ver quanto tempo a gente consegue ficar numa tarefa antes de trocar, e quanto demora para voltar a ela.

---

### 📚 O Conceito

Em 2005, Gloria Mark, Victor González e Justin Harris, da Universidade da Califórnia em Irvine, acompanharam profissionais de escritório com cronômetro na mão, minuto a minuto.

Eles não mediram tarefas pequenas, como responder um e-mail. Mediram o que chamaram de **esfera de trabalho**, uma frente de trabalho com prazo, pessoas e ferramentas próprias, como um projeto, um relatório ou a preparação de uma reunião.

A média de tempo numa esfera de trabalho antes de trocar foi de **11 minutos**. Mas uma média esconde a forma da distribuição. Quando poucos casos muito longos convivem com muitos casos curtos, a média fica acima do que acontece na maior parte das vezes.

Essa é a pergunta do dia. Se a média é 11 minutos, em quantas vezes a pessoa fica mais do que isso?

---

### 🧮 O Modelo

Duas fontes e seis contas.

1. **A forma do tempo numa tarefa.** Uma distribuição lognormal ajustada à média e ao desvio-padrão publicados, para estimar a mediana e a chance de passar de 11 minutos. Intervalo por reamostragem de 2.246 trechos de trabalho (200 mil sorteios no total, semente 42).
2. **Outra forma possível.** O mesmo cálculo com uma distribuição gama, para ver quanto a resposta depende da escolha.
3. **Quando e onde.** A chance de passar de 11 minutos de manhã e à tarde, e para quem trabalha perto ou longe da equipe.
4. **A volta.** A mesma conta para o tempo até retomar uma tarefa interrompida, o número que deu origem ao mito dos "23 minutos".
5. **Quem puxa a volta.** O tempo até retomar quando a pessoa volta por conta própria e quando algo de fora puxa a volta.
6. **Dez anos depois.** A comparação com a atenção na tela medida pela própria Gloria Mark em 2016.

**Fontes:**
- Mark, G., González, V. M. & Harris, J. (2005). No task left behind? Examining the nature of fragmented work. *Proceedings of CHI 2005*, 321-330. **24 profissionais** (7 gerentes, 9 analistas e 8 desenvolvedores), cerca de 25 horas e 42 minutos de observação cada, mais de 700 horas no total. Lido no texto completo.
- Mark, G., Iqbal, S. T., Czerwinski, M., Johns, P. & Sano, A. (2016). Neurotics can't focus: An in situ study of online multitasking in the workplace. *Proceedings of CHI 2016*, 1739-1744. **40 profissionais**, atividade no computador registrada por duas semanas. Lido no resumo.

**Nota metodológica sobre o fator ×0,80.** Não se aplica. As duas fontes são observação cronometrada e registro automático, e não autorrelato.

**Cordas declaradas:** #020 (multitarefa, com outro estudo de Gloria Mark, de 2008) e #163 (a hora do dia). Aqui a pergunta é quanto dura uma tarefa antes da troca.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE, N_SIM = 42, 200_000
rng = np.random.default_rng(SEMENTE)

def jeffreys(k, n, nivel=0.95):
    """Intervalo de Jeffreys para uma proporção."""
    a = (1 - nivel) / 2
    return stats.beta.ppf([a, 1 - a], k + 0.5, n - k + 0.5)

def sorteia(k, n):
    return rng.beta(k + 0.5, n - k + 0.5, N_SIM)

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---

# 1) Mark, González & Harris (2005), CHI 2005, 321-330. Texto completo.
#    Tempos em minutos (média, desvio-padrão), convertidos de minutos e segundos.
mm = lambda m, s: m + s / 60
esfera = dict(media=mm(11, 4), dp=mm(18, 9), trechos=2246)          # tempo numa esfera de trabalho antes de trocar
grupos = {'Perto da equipe': (mm(11, 56), mm(15, 33)), 'Longe da equipe': (mm(9, 56), mm(13, 29)),
          'De manhã': (mm(10, 12), mm(12, 56)), 'À tarde': (mm(14, 40), mm(16, 14))}
interrompidos = dict(k=1282, n=2246)                                  # 57,1% dos trechos interrompidos
retomada = dict(media=mm(25, 26), dp=mm(54, 48), mesmo_dia=0.772, atividades_no_meio=2.26)
retomada_origem = {'Por conta própria': (mm(21, 28), mm(46, 47)), 'Puxada por algo de fora': (mm(61, 37), mm(95, 11))}
retomada_tipo = {'Trabalho principal': 0.820, 'Trabalho secundário': 0.438}   # retomado no mesmo dia
origem_interrupcao = {'da própria pessoa': 0.520, 'de fora': 0.480}
pessoas = 24                                                          # cerca de 25 h 42 min de observação por pessoa

# 2) Mark, Iqbal, Czerwinski, Johns & Sano (2016), CHI 2016, 1739-1744. Resumo.
tela_2016 = dict(pessoas=40, mediana_segundos=40)

aplica_fator_080 = False

fmt = lambda x, c=1: f"{x:.{c}f}".replace('.', ',')
def ms(minutos):
    m = int(minutos); s = int(round((minutos - m) * 60))
    if s == 60: m, s = m + 1, 0
    return f"{m} min {s:02d} s"
print("=" * 72)
print("  DADOS DA LITERATURA - ONZE MINUTOS")
print("=" * 72)
print(f"\n  Mark, González & Harris (2005): {pessoas} profissionais, " + f"{esfera['trechos']:,} trechos de trabalho".replace(',', '.'))
print(f"  → Tempo numa esfera de trabalho: média {ms(esfera['media'])}, desvio-padrão {ms(esfera['dp'])}")
for g, (m, d) in grupos.items():
    print(f"     {g:<16} média {ms(m)}, desvio-padrão {ms(d)}")
print(f"  → Trechos interrompidos: {interrompidos['k']} de {interrompidos['n']} ({fmt(interrompidos['k']/interrompidos['n']*100)}%)")
print(f"  → Retomada no mesmo dia: {fmt(retomada['mesmo_dia']*100)}%  ·  tempo até retomar: média {ms(retomada['media'])}, desvio-padrão {ms(retomada['dp'])}")
print(f"  → Outras atividades antes de retomar: {fmt(retomada['atividades_no_meio'], 2)} em média")
print(f"  → Interrupções {', '.join(f'{k} {fmt(v*100, 0)}%' for k, v in origem_interrupcao.items())}")
for k, (m, d) in retomada_origem.items():
    print(f"  → Retomada {k.lower()}: média {ms(m)}, desvio-padrão {ms(d)}")
for k, v in retomada_tipo.items():
    print(f"  → {k}: {fmt(v*100)}% retomado no mesmo dia")
print(f"\n  Mark et al. (2016): {tela_2016['pessoas']} profissionais  ·  mediana de atenção numa tela: {tela_2016['mediana_segundos']} segundos")
print(f"\n  Fator ×0,80 aplicado: {'sim' if aplica_fator_080 else 'não'}")
print("=" * 72)

In [ ]:
# --- O MODELO ---
LIMITE = 11.0       # minutos

def lognormal(media, dp):
    """Parâmetros da lognormal com a média e o desvio-padrão dados."""
    s2 = np.log(1 + (dp / media) ** 2)
    return np.log(media) - s2 / 2, np.sqrt(s2)

def p_acima_ln(media, dp, x=LIMITE):
    mu, s = lognormal(media, dp)
    return 1 - stats.norm.cdf((np.log(x) - mu) / s)

def mediana_ln(media, dp):
    mu, _ = lognormal(media, dp)
    return np.exp(mu)

def p_acima_gama(media, dp, x=LIMITE):
    k = (media / dp) ** 2; theta = dp ** 2 / media
    return stats.gamma.sf(x, a=k, scale=theta)

# Conta 1 · A forma do tempo numa tarefa, com reamostragem paramétrica dos 2.246 trechos
p11 = p_acima_ln(esfera['media'], esfera['dp'])
med = mediana_ln(esfera['media'], esfera['dp'])
mu, s = lognormal(esfera['media'], esfera['dp'])
REPS = 2000
amostras = np.exp(rng.normal(mu, s, size=(REPS, esfera['trechos'])))
boot_p = np.array([p_acima_ln(a.mean(), a.std(ddof=1)) for a in amostras])
boot_med = np.array([mediana_ln(a.mean(), a.std(ddof=1)) for a in amostras])
ic_p = np.percentile(boot_p, [2.5, 97.5]) * 100
ic_med = np.percentile(boot_med, [2.5, 97.5])
p30 = p_acima_ln(esfera['media'], esfera['dp'], 30)
p3 = 1 - p_acima_ln(esfera['media'], esfera['dp'], 3)

# Conta 2 · Outra forma possível
p11_gama = p_acima_gama(esfera['media'], esfera['dp'])
med_gama = stats.gamma.median(a=(esfera['media'] / esfera['dp']) ** 2, scale=esfera['dp'] ** 2 / esfera['media'])

# Conta 3 · Quando e onde
p_grupos = {g: (p_acima_ln(m, d), mediana_ln(m, d)) for g, (m, d) in grupos.items()}

# Conta 4 · A volta
med_volta = mediana_ln(retomada['media'], retomada['dp'])
p_volta_25 = p_acima_ln(retomada['media'], retomada['dp'], retomada['media'])
p_volta_60 = p_acima_ln(retomada['media'], retomada['dp'], 60)
p_int = interrompidos['k'] / interrompidos['n']; ic_int = jeffreys(interrompidos['k'], interrompidos['n'])

med_origem = {k: mediana_ln(m, d) for k, (m, d) in retomada_origem.items()}
razao_origem = med_origem['Puxada por algo de fora'] / med_origem['Por conta própria']

# Conta 5 · Dez anos depois (medidas diferentes, só como ordem de grandeza)
razao_2005_2016 = med * 60 / tela_2016['mediana_segundos']

print("=" * 72)
print("  MODELO - RESULTADO")
print("=" * 72)
print("\n  CONTA 1 · A forma do tempo numa tarefa (lognormal)")
print(f"  → Mediana: {ms(med)} [{ms(ic_med[0])}; {ms(ic_med[1])}]  ·  média publicada: {ms(esfera['media'])}")
print(f"  → P(passar de 11 minutos) = {fmt(p11*100)}% [{fmt(ic_p[0])}; {fmt(ic_p[1])}]")
print(f"  → P(passar de 30 minutos) = {fmt(p30*100)}%  ·  P(menos de 3 minutos) = {fmt(p3*100)}%")
print("\n  CONTA 2 · Outra forma possível (gama)")
print(f"  → P(passar de 11 minutos) = {fmt(p11_gama*100)}%  ·  mediana {ms(med_gama)}")
print("\n  CONTA 3 · Quando e onde")
for g, (p, m) in p_grupos.items():
    print(f"  → {g:<16} P(passar de 11 minutos) = {fmt(p*100)}%  ·  mediana {ms(m)}")
print("\n  CONTA 4 · A volta")
print(f"  → Trechos interrompidos: {fmt(p_int*100)}% [{fmt(ic_int[0]*100)}; {fmt(ic_int[1]*100)}]")
print(f"  → Tempo até retomar: mediana {ms(med_volta)}, contra a média de {ms(retomada['media'])}")
print(f"  → P(demorar mais que a média) = {fmt(p_volta_25*100)}%  ·  P(demorar mais de 1 hora) = {fmt(p_volta_60*100)}%")
for k, v in med_origem.items():
    print(f"  → Volta {k.lower()}: mediana {ms(v)}")
print(f"  → Quando algo de fora puxa a volta, a mediana é {fmt(razao_origem, 1)} vezes maior")
print("\n  CONTA 5 · Dez anos depois")
print(f"  → 2005, mediana numa esfera de trabalho: {ms(med)}  ·  2016, mediana numa tela: {tela_2016['mediana_segundos']} s")
print(f"  → A primeira é cerca de {fmt(razao_2005_2016, 0)} vezes a segunda (medidas diferentes)")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fonte: Mark, González & Harris (2005), CHI 2005. Conta própria | #365Probabilidades'

# GRÁFICO 1 · A forma do tempo numa tarefa
fig1, ax1 = plt.subplots(figsize=(12, 8))
xs = np.linspace(0.05, 45, 800); dens = stats.lognorm.pdf(xs, s=s, scale=np.exp(mu))
ax1.plot(xs, dens, color=VERDE, linewidth=2.5)
ax1.fill_between(xs, dens, where=xs >= LIMITE, color=DOURADO, alpha=0.45)
ax1.axvline(med, color=VERDE, linestyle='--', linewidth=2); ax1.axvline(esfera['media'], color=VERMELHO, linestyle='--', linewidth=2)
ax1.text(med + 0.4, dens.max() * 0.92, f"mediana\n{ms(med)}", color=VERDE, fontsize=12, fontweight='bold')
ax1.text(esfera['media'] + 0.4, dens.max() * 0.72, f"média\n{ms(esfera['media'])}", color=VERMELHO, fontsize=12, fontweight='bold')
ax1.text(20, dens.max() * 0.28, f"{fmt(p11*100, 0)}% das vezes\npassa de 11 minutos", fontsize=15, fontweight='bold', color='#8a6d1f')
ax1.set_xlim(0, 45); ax1.set_ylim(0, dens.max() * 1.05)
ax1.set_xlabel('Minutos numa tarefa antes de trocar'); ax1.set_ylabel('Densidade')
ax1.set_title('A média é 11 minutos, mas na maior parte das vezes a troca vem antes\nDistribuição lognormal ajustada aos números publicados', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-165-grafico-01-a-forma.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · Quando e onde
fig2, ax2 = plt.subplots(figsize=(12, 8))
rot = ['Todos', 'Todos\n(forma gama)'] + list(p_grupos)
vals = [p11 * 100, p11_gama * 100] + [p * 100 for p, _ in p_grupos.values()]
cores = [DOURADO, CINZA, VERDE, VERDE, VERMELHO, VERMELHO]
b2 = ax2.bar(rot, vals, color=cores, width=0.6)
ax2.errorbar(0, p11 * 100, yerr=[[p11 * 100 - ic_p[0]], [ic_p[1] - p11 * 100]], color='black', capsize=6)
for i, (b, v) in enumerate(zip(b2, vals)):
    topo = ic_p[1] if i == 0 else v
    ax2.text(b.get_x() + b.get_width()/2, topo + 1.5, f"{fmt(v, 0)}%", ha='center', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 50); ax2.set_ylabel('Chance de passar de 11 minutos numa tarefa (%)'); ax2.tick_params(axis='x', labelsize=12)
ax2.set_title('À tarde e perto da equipe, a tarefa dura um pouco mais\nChance de passar de 11 minutos, por grupo', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-165-grafico-02-quando-e-onde.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · A volta
fig3, ax3 = plt.subplots(figsize=(12, 8))
mu_v, s_v = lognormal(retomada['media'], retomada['dp'])
xv = np.linspace(0.05, 90, 900); dv = stats.lognorm.pdf(xv, s=s_v, scale=np.exp(mu_v))
ax3.plot(xv, dv, color=VERDE, linewidth=2.5); ax3.fill_between(xv, dv, color=VERDE, alpha=0.15)
ax3.axvline(med_volta, color=VERDE, linestyle='--', linewidth=2); ax3.axvline(retomada['media'], color=VERMELHO, linestyle='--', linewidth=2)
ax3.text(med_volta + 0.8, dv.max() * 0.9, f"mediana\n{ms(med_volta)}", color=VERDE, fontsize=12, fontweight='bold')
ax3.text(retomada['media'] + 0.8, dv.max() * 0.7, f"média\n{ms(retomada['media'])}", color=VERMELHO, fontsize=12, fontweight='bold')
ax3.text(45, dv.max() * 0.4, f"Metade das retomadas acontece\nem menos de {fmt(med_volta, 0)} minutos.\nA média sobe por causa\nde poucos casos muito longos.", fontsize=12)
ax3.set_xlim(0, 90); ax3.set_ylim(0, dv.max() * 1.05)
ax3.set_xlabel('Minutos até voltar a uma tarefa interrompida (retomadas no mesmo dia)'); ax3.set_ylabel('Densidade')
ax3.set_title('O tempo de volta mede quanto se demora para retomar, e não para se concentrar\nDistribuição lognormal ajustada aos números publicados', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-165-grafico-03-a-volta.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · Quem puxa a volta
fig5, ax5 = plt.subplots(figsize=(12, 8))
rot5 = list(med_origem); v5 = list(med_origem.values()); m5 = [retomada_origem[k][0] for k in rot5]
xs5 = np.arange(2); lg = 0.36
b5a = ax5.bar(xs5 - lg/2, v5, lg, color=VERDE, label='Mediana (estimada)')
b5b = ax5.bar(xs5 + lg/2, m5, lg, color=CINZA, alpha=0.6, label='Média (publicada)')
for b in list(b5a) + list(b5b):
    ax5.text(b.get_x() + b.get_width()/2, b.get_height() + 1, ms(b.get_height()), ha='center', fontsize=12, fontweight='bold')
ax5.set_xticks(xs5); ax5.set_xticklabels(['Voltou por conta própria', 'Voltou porque algo de fora puxou'], fontsize=13)
ax5.set_ylabel('Minutos até voltar à tarefa interrompida'); ax5.set_ylim(0, 75); ax5.legend(frameon=False, loc='upper left')
ax5.set_title('Quando algo de fora puxa a volta, ela demora mais\nTempo até retomar uma tarefa interrompida, no mesmo dia', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-165-grafico-04-quem-puxa-a-volta.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# GRÁFICO 5 · Dez anos depois
fig4, ax4 = plt.subplots(figsize=(12, 8))
rot4 = ['2005\nnuma frente de trabalho\n(24 profissionais)', '2016\nnuma tela do computador\n(40 profissionais)']
v4 = [med * 60, tela_2016['mediana_segundos']]
b4 = ax4.bar(rot4, v4, color=[DOURADO, VERMELHO], width=0.45)
for b, v, t in zip(b4, v4, [ms(med) + ' (estimada)', f"{tela_2016['mediana_segundos']} s"]):
    ax4.text(b.get_x() + b.get_width()/2, v + 8, t, ha='center', fontsize=16, fontweight='bold')
ax4.set_ylabel('Mediana (segundos)'); ax4.set_ylim(0, med * 60 * 1.2); ax4.tick_params(axis='x', labelsize=12)
ax4.set_title('Dez anos depois, a atenção numa tela dura menos de um minuto\nMedianas de dois estudos de Gloria Mark, que medem coisas diferentes', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, 'Fontes: Mark, González & Harris (2005); Mark, Iqbal, Czerwinski, Johns & Sano (2016). Conta própria | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-165-grafico-05-dez-anos-depois.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

### 💡 O Insight

**A média é de 11 minutos numa tarefa antes de trocar. Mas só em 29% das vezes a tarefa passa de 11 minutos.**

Na maior parte das vezes, a troca vem bem antes. Pela distribuição ajustada aos números publicados, metade das tarefas dura menos de 6 minutos. A média fica em 11 porque algumas tarefas longas, de meia hora ou mais, puxam a conta para cima.

À tarde, a chance de passar de 11 minutos sobe para 45%. De manhã, fica em 29%.

Pouco mais da metade das interrupções (52%) veio da própria pessoa. E quem voltou à tarefa por conta própria voltou em cerca de 9 minutos, na mediana, contra cerca de 33 minutos quando algo de fora puxou a volta.

O número dos "25 minutos" também é real, e mede outra coisa. É o tempo até a pessoa voltar a uma tarefa interrompida, depois de passar, em média, por outras duas. Metade dessas voltas acontece em menos de 11 minutos. A frase que circula, de que se leva 23 minutos para recuperar a concentração, não está no artigo.

Dez anos depois, a mesma pesquisadora mediu a atenção numa tela do computador, e a mediana foi de 40 segundos. As medidas são diferentes, mas apontam na mesma direção.

A pergunta que fica:

*Qual foi a última tarefa em que você ficou mais de 11 minutos sem trocar?*

---

### ⚠️ Limitações do Modelo

- O estudo de 2005 tem 24 pessoas de duas equipes de uma mesma empresa, observadas por cerca de três dias e meio cada, antes do smartphone.
- A mediana e a chance de passar de 11 minutos não estão no artigo. Elas vêm de uma distribuição lognormal ajustada à média e ao desvio-padrão publicados. Com uma distribuição gama, a chance de passar de 11 minutos quase não muda (29%), mas a mediana cai para cerca de 3 minutos e 40 segundos. O número de 29% é robusto à escolha. A mediana não é.
- O número de 2.246 trechos de trabalho foi deduzido dos graus de liberdade de um teste do artigo, e a contagem de 1.282 trechos interrompidos é coerente com os 57,1% publicados. O intervalo da Conta 1 trata os trechos como independentes, mas eles vêm de só 24 pessoas, e por isso o intervalo é mais estreito do que deveria.
- A esfera de trabalho é uma unidade definida pelos pesquisadores, maior do que uma tarefa pequena. O título fala em "tarefa" para simplificar.
- A divisão entre volta por conta própria e volta puxada de fora segue a definição do artigo. As medianas dessas duas voltas também vêm da distribuição ajustada, e não do artigo.
- O tempo até retomar vale só para as tarefas retomadas no mesmo dia (77,2%). As outras 22,8% ficaram para depois e não entram na conta.
- O estudo de 2016 foi lido no resumo. Ele mede o tempo numa tela do computador, registrado automaticamente, e não a esfera de trabalho. A comparação entre 2005 e 2016 vale como ordem de grandeza, e não como medida da mesma coisa.
- Os "23 minutos para recuperar a concentração" vêm de uma entrevista, e não do artigo. O projeto já tratou desse número no Dia #020.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*